In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

Map(center=[8.5, -80], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [ ]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Panama Road Network 
#### OpenStreetMap-derived by STRI: https://stridata-si.opendata.arcgis.com/datasets/dd8a5fe5524a466cad3c584d6375a72f/about
#### Imported to GEE code editor


In [ ]:
roads = ee.FeatureCollection("projects/deforestation-495419/assets/Panama_OSM_Roads")

# Rasterize roads
roads_raster = ee.Image().byte().paint(roads, 1)

# Distance to nearest road in meters
distance_to_roads = (
    roads_raster
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(30)  # adjust if your dataset resolution differs
    .rename("dist_roads_m")
    .clip(panama_geom)
)

vis = {
    "min": 0,
    "max": 5000,
    "palette": ["white", "blue", "green", "yellow", "red"]
}

Map.addLayer(panama_geom, {}, "Panama boundary")

Map.addLayer(
    roads_raster,
    {"palette": ["black"]},
    "Roads (raster)"
)

Map.addLayer(
    distance_to_roads,
    vis,
    "Distance to Roads (m)"
)

In [ ]:
# Display map
Map.centerObject(panama_geom, 7)
Map